In [1]:
!pip install -q kagglehub

import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
path = kagglehub.dataset_download("pythonafroz/medquad-medical-question-answer-for-ai-research")
print("Downloaded to:", path)

100%|██████████| 4.95M/4.95M [00:00<00:00, 73.4MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research/versions/1


In [3]:
import os
import pandas as pd

csv_candidates = [os.path.join(root, f) for root, _, files in os.walk(path) for f in files if f.endswith(".csv")]
print("CSV files found:", csv_candidates)

df = pd.read_csv(csv_candidates[0])
print()
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print()
print(df.head(3).to_string())

CSV files found: ['/root/.cache/kagglehub/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research/versions/1/medquad.csv']

Shape: (16412, 4)
Columns: ['question', 'answer', 'source', 'focus_area']

                              question                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [4]:
def find_column(df, keywords):
    matches = [c for c in df.columns if any(k.lower() in c.lower() for k in keywords)]
    if not matches:
        raise ValueError(f"No column found matching {keywords} among {df.columns.tolist()}")
    if len(matches) > 1:
        print(f"WARNING: multiple columns matched {keywords}: {matches} — using the first one, verify this is right")
    return matches[0]

question_col = find_column(df, ["question"])
answer_col = find_column(df, ["answer"])
print("Question column:", question_col)
print("Answer column:", answer_col)

print()
print("Nulls in question:", df[question_col].isna().sum())
print("Nulls in answer:", df[answer_col].isna().sum())

df = df.dropna(subset=[question_col, answer_col]).reset_index(drop=True)
print("After dropping nulls:", df.shape)

Question column: question
Answer column: answer

Nulls in question: 0
Nulls in answer: 5
After dropping nulls: (16407, 4)


In [5]:
!pip install -q nltk

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

import re
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

_stemmer = PorterStemmer()
_stop_words = set(stopwords.words("english"))

def stem_and_clean(text: str) -> str:
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text.lower())
    tokens = word_tokenize(text)
    stemmed = [_stemmer.stem(t) for t in tokens if len(t) > 1 and t not in _stop_words]
    return " ".join(stemmed)

# Sanity check the exact bug this fixes, before trusting it on the real corpus
test = stem_and_clean("How do I treat my migraines?")
print("Preprocessed test string:", repr(test))

Preprocessed test string: 'treat migrain'


In [21]:
dupe_count = df.duplicated(subset=[question_col]).sum()
print(f"Duplicate questions: {dupe_count} out of {len(df)}")

df = df.drop_duplicates(subset=[question_col]).reset_index(drop=True)
print(f"After deduplication: {df.shape}")

Duplicate questions: 0 out of 14979
After deduplication: (14979, 4)


In [22]:
# Deduplicate first (from the earlier finding)
dupe_count = df.duplicated(subset=[question_col]).sum()
print(f"Duplicate questions: {dupe_count}")
df = df.drop_duplicates(subset=[question_col]).reset_index(drop=True)

# Re-run preprocessing + fit on the deduplicated corpus
questions = df[question_col].astype(str).tolist()
answers = df[answer_col].astype(str).tolist()
preprocessed_questions = [stem_and_clean(q) for q in questions]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(preprocessed_questions)
print("Final corpus size:", tfidf_matrix.shape)

Duplicate questions: 0
Final corpus size: (14979, 4348)


In [23]:
questions = df[question_col].astype(str).tolist()
answers = df[answer_col].astype(str).tolist()

preprocessed_questions = [stem_and_clean(q) for q in questions]

print("Example before/after:")
for i in [0, 1, 2]:
    print(f"  original:     {questions[i]!r}")
    print(f"  preprocessed: {preprocessed_questions[i]!r}")
    print()

Example before/after:
  original:     'What is (are) Glaucoma ?'
  preprocessed: 'glaucoma'

  original:     'What causes Glaucoma ?'
  preprocessed: 'caus glaucoma'

  original:     'What are the symptoms of Glaucoma ?'
  preprocessed: 'symptom glaucoma'



In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

# IMPORTANT: fit on the already-preprocessed text with TfidfVectorizer's
# OWN default tokenizer — never pass tokenizer=stem_and_clean here. A
# vectorizer fit with a custom Python callable can't be safely loaded back
# in a different process (the service) — confirmed this fails with
# AttributeError on load. Pre-stemming the text avoids the issue entirely.
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(preprocessed_questions)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

TF-IDF matrix shape: (14979, 4348)
Vocabulary size: 4348


In [25]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def test_retrieve(query, top_k=3):
    cleaned = stem_and_clean(query)
    query_vec = vectorizer.transform([cleaned])
    sims = cosine_similarity(query_vec, tfidf_matrix)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    for i in top_idx:
        print(f"  score={sims[i]:.3f}  Q: {questions[i]}")

print("Test 1: plural/verb-form robustness")
test_retrieve("How do I treat my migraines?")

print()
print("Test 2: a completely different real question")
test_retrieve("What are the symptoms of diabetes?")

print()
print("Test 3: off-topic — expect low scores across the board")
test_retrieve("what is the capital of France")

Test 1: plural/verb-form robustness
  score=0.675  Q: What is (are) Migraine ?
  score=0.626  Q: What are the treatments for Migraine ?
  score=0.564  Q: What is the outlook for Migraine ?

Test 2: a completely different real question
  score=1.000  Q: What are the symptoms of Diabetes ?
  score=0.893  Q: What is (are) Diabetes ?
  score=0.787  Q: What are the treatments for Diabetes ?

Test 3: off-topic — expect low scores across the board
  score=0.510  Q: What are the symptoms of Slipped capital femoral epiphysis ?
  score=0.000  Q: Is TARP syndrome inherited ?
  score=0.000  Q: What causes TARP syndrome ?


In [26]:
qa_corpus = [
    {"question": questions[i], "answer": answers[i], "sources": ["MedQuAD"]}
    for i in range(len(questions))
]
print(f"{len(qa_corpus)} entries ready for export (post-deduplication)")
print(qa_corpus[0])

assert tfidf_matrix.shape[0] == len(qa_corpus), "row count mismatch — do not export"
print("Row count confirmed matching between matrix and corpus.")

14979 entries ready for export (post-deduplication)
{'question': 'What is (are) Glaucoma ?', 'answer': "Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. While glaucoma can strike anyone, the risk is much greater for people over 60. How Glaucoma Develops  There are several different types of glaucoma. Most of these involve the drainage system within the eye. At the front of the eye there is a small space called the anterior chamber. A clear fluid flows through this chamber and bathes and nourishes the nearby tissues. (Watch the video to learn more about glaucoma. To enlarge the video, click the brackets in the lower right-hand corner. To reduce the video, press the Escape (Esc) button on your keyboard.) In glaucoma, for still unknown reasons, the fluid drains too slowly out of the eye. As the fluid builds up, the pressure inside the eye rises. Unless this pressure is controlled, it may cause damage to the optic nerve and othe

In [27]:
import joblib
import json
import os

os.makedirs("chatbot_artifacts", exist_ok=True)

joblib.dump(vectorizer, "chatbot_artifacts/tfidf_vectorizer.joblib")
joblib.dump(tfidf_matrix, "chatbot_artifacts/tfidf_matrix.joblib")
with open("chatbot_artifacts/qa_corpus.json", "w") as f:
    json.dump(qa_corpus, f)

print("Artifacts written:")
!ls -la chatbot_artifacts

Artifacts written:
total 21340
drwxr-xr-x 2 root root     4096 Sep 27 19:32 .
drwxr-xr-x 1 root root     4096 Sep 27 19:32 ..
-rw-r--r-- 1 root root 20958306 Sep 27 19:50 qa_corpus.json
-rw-r--r-- 1 root root   782971 Sep 27 19:50 tfidf_matrix.joblib
-rw-r--r-- 1 root root    94723 Sep 27 19:50 tfidf_vectorizer.joblib


In [28]:
import shutil
shutil.make_archive("v1_chatbot_artifacts", "zip", "chatbot_artifacts")

from google.colab import files
files.download("v1_chatbot_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>